# Transfer learning
Our models were trained on MNIST. Real handwriting looks different, so here I fine-tune them on other handwriting
datasets and compare before and after.

Datasets (all listed in `preprocessing.DATASETS`):
- **mnist**: the original training data
- **swiss**: Swiss handwritten digits (Kaggle, olafkrastovski, CC0)
- **math**: Handwritten Math Symbols (Kaggle, sagyamthapa), digits plus + - * /

`transfer.py` builds the training mixes, fine-tunes and evaluates, so a new dataset only needs a loader in `preprocessing.py`.

In [ ]:
import sys
sys.path.insert(0, "..")

import os, glob
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from hnrs.preprocessing import load_dataset, CLASSES
from hnrs.transfer import build_mix, fine_tune, compare, match_input
from hnrs.models.lenet import build_model, MODEL_PATH, FINETUNED_PATH, MIXED_PATH, SYMBOLS_PATH
from hnrs.preprocess_image import preprocess_image
from hnrs.arithmetic import calculate

CONV2D_PATH = "../models/conv2d.keras"
DENSE_PATH = "../models/dense.keras"

DIGIT_SETS = ["mnist", "swiss"]
ALL_SETS = ["mnist", "swiss", "math"]

swiss_x_train, swiss_y_train, swiss_x_test, swiss_y_test = load_dataset("swiss")
print(swiss_x_train.shape, swiss_x_test.shape)

## 1. Original models on each dataset
All three were trained on MNIST only. For math this only counts the digits, since these models can't predict symbols.

In [ ]:
original = tf.keras.models.load_model(MODEL_PATH)
originals = {
    "LeNet": original,
    "Conv2D": tf.keras.models.load_model(CONV2D_PATH),
    "Dense": tf.keras.models.load_model(DENSE_PATH),
}
compare(originals, ALL_SETS)

original_pred = original.predict(match_input(original, swiss_x_test), verbose=0).argmax(axis=1)
print(classification_report(swiss_y_test, original_pred, digits=4))

## 2. Fine-tune LeNet on Swiss: freeze the conv layers, retrain the dense layers

In [ ]:
x, y = build_mix(["swiss"])
finetuned, _ = fine_tune(MODEL_PATH, x, y, save_path=FINETUNED_PATH)
compare({"Fine-tuned (Swiss)": finetuned}, DIGIT_SETS)

finetuned_pred = finetuned.predict(match_input(finetuned, swiss_x_test), verbose=0).argmax(axis=1)
print(classification_report(swiss_y_test, finetuned_pred, digits=4))

## 2b. Fine-tune on a mix of MNIST and Swiss
Fine-tuning on Swiss alone made the model misread my 7s (no crossbar) as 1s, because Swiss 1s have a long flag.
Mixing in the same amount of MNIST should let it learn both styles.

In [ ]:
x, y = build_mix(["mnist", "swiss"])
mixed, _ = fine_tune(MODEL_PATH, x, y, save_path=MIXED_PATH)
compare({"Fine-tuned (mixed)": mixed}, DIGIT_SETS)

## 3. Accuracy per digit, before vs after

In [ ]:
before = [(original_pred[swiss_y_test == d] == d).mean() for d in range(10)]
after = [(finetuned_pred[swiss_y_test == d] == d).mean() for d in range(10)]

x_pos = np.arange(10)
plt.bar(x_pos - 0.2, before, width=0.4, label="Original")
plt.bar(x_pos + 0.2, after, width=0.4, label="Fine-tuned")
plt.xticks(x_pos)
plt.ylim(0.6, 1.0)
plt.xlabel("Digit")
plt.ylabel("Accuracy on Swiss test set")
plt.legend(loc="lower right")
plt.savefig("../docs/images/lenet_transfer_per_digit.png", dpi=150)
plt.show()

## 4. Does transfer learning help more when there's less data?
Train on only n Swiss images, either from scratch or starting from the MNIST LeNet.

In [ ]:
def from_scratch(n):
    model = build_model()
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    model.fit(match_input(model, swiss_x_train[:n]), swiss_y_train[:n], epochs=5, verbose=0)
    return model.evaluate(match_input(model, swiss_x_test), swiss_y_test, verbose=0)[1]

def with_transfer(n):
    model, _ = fine_tune(MODEL_PATH, swiss_x_train[:n], swiss_y_train[:n])
    return model.evaluate(match_input(model, swiss_x_test), swiss_y_test, verbose=0)[1]

print("train size | from scratch | transfer")
for n in [200, 1000, len(swiss_x_train)]:
    print(f"{n:10d} | {from_scratch(n):.4f}       | {with_transfer(n):.4f}")

## 5. My own handwriting
The fully independent test: photos I wrote and took myself, never used for training.

In [ ]:
# My own handwriting photos, named by label: 0_1.jpg, 0_2.jpg, 1_1.jpg ...
paths = sorted(glob.glob("../data/handwritten/*"))
paths = [p for p in paths if p.lower().endswith((".jpg", ".jpeg", ".png"))]

names, labels, digits = [], [], []
for path in paths:
    name = os.path.basename(path)
    if not name[0].isdigit():
        print(f"{name}: not named by label, skipped")
        continue
    try:
        digits.append(preprocess_image(path))
    except ValueError:
        print(f"{name}: no digit found, skipped")
        continue
    names.append(name)
    labels.append(int(name[0]))

photos = np.array(digits)
labels = np.array(labels)

def photo_predictions(model):
    """
    returns the model's predictions for my photos
    """
    return model.predict(match_input(model, photos), verbose=0).argmax(axis=1)

original_photo_pred = photo_predictions(original)
finetuned_photo_pred = photo_predictions(finetuned)
mixed_photo_pred = photo_predictions(mixed)

print(f"{len(labels)} photos")
print(f"Original LeNet:   {(original_photo_pred == labels).mean():.1%}")
print(f"Fine-tuned LeNet: {(finetuned_photo_pred == labels).mean():.1%}")
print(f"Mixed LeNet:      {(mixed_photo_pred == labels).mean():.1%}")

# Which photos did each model get wrong?
for name, true, a, b, c in zip(names, labels, original_photo_pred, finetuned_photo_pred, mixed_photo_pred):
    if a != true or b != true or c != true:
        print(f"  {name}: true {true}, original {a}, fine-tuned {b}, mixed {c}")

In [ ]:
# Grid: preprocessed photo, with original > fine-tuned > mixed prediction
# Green = mixed model correct, red = mixed model wrong
cols = 10
rows = int(np.ceil(len(labels) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.2, rows * 1.5))
for ax in np.array(axes).flat:
    ax.axis("off")
for ax, digit, true, a, b, c in zip(np.array(axes).flat, digits, labels, original_photo_pred, finetuned_photo_pred, mixed_photo_pred):
    ax.imshow(digit, cmap="gray")
    ax.set_title(f"{a} > {b} > {c}", color="green" if c == true else "red", fontsize=7)
plt.tight_layout()
plt.savefig("../docs/images/handwritten_transfer.png", dpi=150)
plt.show()

## 6. Adding arithmetic symbols
Extension option 2. Start from the mixed model, add the math dataset, and the output layer grows from 10 to 14
(0-9 and + - * /). MNIST and Swiss stay in the mix so it doesn't forget their digits.

In [ ]:
x, y = build_mix(["mnist", "swiss", "math"])
symbols, _ = fine_tune(MIXED_PATH, x, y, save_path=SYMBOLS_PATH)
compare({"Symbols model": symbols}, ALL_SETS)

_, _, math_x_test, math_y_test = load_dataset("math")
math_pred = symbols.predict(match_input(symbols, math_x_test), verbose=0).argmax(axis=1)
print(classification_report(math_y_test, math_pred, target_names=CLASSES, digits=3))

ConfusionMatrixDisplay.from_predictions(math_y_test, math_pred, display_labels=CLASSES, colorbar=False)
plt.title("Symbols model confusion matrix")
plt.show()

Predict a few expressions one character at a time and calculate the result.
This skips segmentation, so it only tests recognition and calculation.

In [ ]:
rng = np.random.default_rng(42)

def pick_image(char):
    """
    returns a random math test image of the given character
    """
    label = CLASSES.index(char)
    return math_x_test[rng.choice(np.where(math_y_test == label)[0])]

for expression in ["12+3*4", "9-4/2", "7*8", "100/25+6"]:
    images = np.array([pick_image(c) for c in expression])
    pred = symbols.predict(match_input(symbols, images), verbose=0).argmax(axis=1)
    predicted = "".join(CLASSES[i] for i in pred)
    try:
        result = calculate(predicted)
    except ValueError as e:
        result = f"error: {e}"
    print(f"actual {expression} = {calculate(expression)} | predicted {predicted} = {result}")

## 7. Same fine-tuning for every model
The original Conv2D beat the original LeNet on Swiss, so to compare fairly every model gets the same MNIST + Swiss mix.

In [ ]:
x, y = build_mix(["mnist", "swiss"])

tuned = {}
for name, path in [("LeNet", MODEL_PATH), ("Conv2D", CONV2D_PATH), ("Dense", DENSE_PATH)]:
    tuned[name], _ = fine_tune(path, x, y)

compare(tuned, DIGIT_SETS)
for name, model in tuned.items():
    print(f"{name}: {model.count_params():,} parameters, my photos {(photo_predictions(model) == labels).mean():.1%}")

## 8. Summary

In [ ]:
models = {"Original": original, "Fine-tuned (Swiss)": finetuned, "Fine-tuned (mixed)": mixed, "Symbols model": symbols}
compare(models, ALL_SETS)

print()
for name, model in models.items():
    print(f"{name:22s} my photos {(photo_predictions(model) == labels).mean():.1%}")